In [1]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime, to_date
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StructField,
    StructType,
)

In [2]:
def get_config():
    return {
        "minio_root_user": os.environ.get("MINIO_ROOT_USER", "minio"),
        "minio_root_password": os.environ.get("MINIO_ROOT_PASSWORD", "miniominio"),
        "minio_endpoint": os.environ.get(
            "SPARK_MINIO_ENDPOINT",
            "http://urbangreen-minio:9000",
        ),
        "kafka_topic": os.environ.get(
            "KAFKA_TOPIC_SENSOR_READINGS",
            "sensor_readings",
        ),
        "kafka_bootstrap": os.environ.get(
            "SIMULATOR_KAFKA_BOOTSTRAP",
            "urbangreen-kafka:9092",
        ),
        "kafka_offset": os.environ.get(
            "SPARK_KAFKA_OFFSET",
            "earliest",
        ),
        "bucket": os.environ.get(
            "MINIO_STAGING_BUCKET",
            "staging",
        ),
        "trigger_interval": os.environ.get(
            "STREAM_TRIGGER_INTERVAL",
            "60 seconds",
        ),
    }

In [3]:
def create_spark_session(config):
    return (
        SparkSession.builder.appName("Sensor Readings Stream")
        .config(
            "spark.hadoop.fs.s3a.endpoint",
            config["minio_endpoint"],
        )
        .config(
            "spark.hadoop.fs.s3a.access.key",
            config["minio_root_user"],
        )
        .config(
            "spark.hadoop.fs.s3a.secret.key",
            config["minio_root_password"],
        )
        .config(
            "spark.hadoop.fs.s3a.path.style.access",
            "true",
        )
        .config(
            "spark.hadoop.fs.s3a.connection.ssl.enabled",
            "false",
        )
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
        )
        # .config(
        #     "spark.jars.packages",
        #     ",".join(
        #         [
        #             "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2",
        #             "org.apache.hadoop:hadoop-aws:3.4.1",
        #             "com.amazonaws:aws-java-sdk-bundle:1.12.782",
        #         ]
        #     ),
        # )
        .getOrCreate()
    )

In [4]:
def sensor_schema():
    return StructType(
        [
            StructField("farm_sensor_id", IntegerType(), True),
            StructField("farm_id", IntegerType(), True),
            StructField("sensor_type_id", IntegerType(), True),
            StructField("value", DoubleType(), True),
            StructField("timestamp", LongType(), True),
        ]
    )

In [5]:
def read_kafka_stream(spark, config):
    return (
        spark.readStream.format("kafka")
        .option(
            "kafka.bootstrap.servers",
            config["kafka_bootstrap"],
        )
        .option(
            "subscribe",
            config["kafka_topic"],
        )
        .option(
            "startingOffsets",
            config["kafka_offset"],
        )
        .option(
            "failOnDataLoss",
            "false",
        )
        .load()
    )

In [6]:
def transform_sensor_readings(df):

    raw_df = df.select(col("value").cast("string").alias("value"))

    parsed_df = raw_df.withColumn(
        "data",
        from_json(
            col("value"),
            sensor_schema(),
        ),
    ).select("data.*")

    return parsed_df.withColumn(
        "event_date",
        to_date(from_unixtime(col("timestamp"))),
    )

In [7]:
def write_stream(df, config):

    return (
        df.writeStream.format("parquet")
        .outputMode("append")
        .partitionBy("event_date")
        .option(
            "path",
            f"s3a://{config['bucket']}/raw/kafka/{config['kafka_topic']}/",
        )
        .option(
            "checkpointLocation",
            f"s3a://{config['bucket']}/_checkpoints/kafka/{config['kafka_topic']}/",
        )
        .trigger(processingTime=config["trigger_interval"])
        .start()
    )

In [8]:
def main():

    config = get_config()

    spark = create_spark_session(config)

    kafka_df = read_kafka_stream(
        spark,
        config,
    )

    transformed_df = transform_sensor_readings(kafka_df)

    query = write_stream(
        transformed_df,
        config,
    )

    query.awaitTermination()


if __name__ == "__main__":
    main()

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5c96df68-b6c4-4c31-8495-40d86f4181fe;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.2 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala

Py4JJavaError: An error occurred while calling o86.start.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2737)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3569)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.datasources.DataSource.makeQualified(DataSource.scala:125)
	at org.apache.spark.sql.execution.datasources.DataSource.createSink(DataSource.scala:332)
	at org.apache.spark.sql.classic.DataStreamWriter.createV1Sink(DataStreamWriter